# Tome 15, Chapter 11: Problems to work on (Python companion)

One computation per open problem of the chapter, each a starting point rather than an answer: repeated binomial coefficients (Propositions 8.1 and 8.2), lonely runners (Proposition 8.3), graceful trees (Proposition 8.5), Kelly's lemma (Lemma 8.6), Lehmer's problem (Proposition 8.8), Brocard's problem (Proposition 8.9) and Sendov's conjecture (Proposition 8.11). The computer-science application is **backtracking search with pruning**, used to find graceful labellings of all small trees. Exercises first, solutions at the end. Requires `numpy`, `sympy` and `networkx`.

In [1]:
import math, sys, numpy as np, networkx as nx
sys.set_int_max_str_digits(0)
from math import comb, isqrt, gcd
from fractions import Fraction as Fr
from collections import Counter
rng = np.random.default_rng(11)

## Exercises

**P11.1 (★) Repeated binomial coefficients.** List all $a\le10^{15}$ with $N(a)\ge6$, and check Proposition 8.2 for $i=1,\dots,5$.

**P11.2 (★★) Lonely runners.** For every set of $k\le4$ distinct speeds in $\{1,\dots,12\}$, find a time $t$ at which all runners are at distance at least $\frac1{k+1}$ from the stationary one. (Search among the times $t=\frac m{v_i+v_j}$: why are they natural?) Which speed sets are tight?

**P11.3 (★★) Graceful trees by backtracking.** Write a backtracking search for graceful labellings and run it on all trees with at most $11$ vertices. Find a labelling of the spider of Exercise 8.4.

**P11.4 (★) Kelly's lemma.** For random graphs, compute the deck and check Lemma 8.6 for edges, triangles and paths with two edges.

**P11.5 (★★) Lehmer and Carmichael.** Search for composite $n\le10^7$ with $\varphi(n)\mid n-1$. List the Carmichael numbers in that range (squarefree composite $n$ with $p-1\mid n-1$ for all $p\mid n$), and for each compute $(n-1)/\varphi(n)$ as a fraction: how close do they come?

**P11.6 (★) Brocard.** Find all $n\le2000$ with $n!+1$ a square. Check Proposition 8.9 for all primes $p<200$.

**P11.7 (★★) Sendov, numerically.** For random polynomials of degree $n$ with zeros in the unit disc, compute the largest distance from a zero to its nearest critical point. Try zeros uniform in the disc, zeros on the circle, and $z^n-1$ perturbed. Compare with the bound $2n^{-1/(n-1)}$ of Proposition 8.11.

## Solutions

**P11.1.** Every $a\ge3$ appears at $(a,1)$ and $(a,a-1)$; we add the positions with $2\le k\le n-2$, enumerating $k\le n/2$ (at most $\log_2 a$ values of $k$, by the proof of Proposition 8.1).

In [2]:
X = 10**15
cnt = Counter()
k = 2
while comb(2*k, k) <= X:
    n = 2*k
    while True:
        a = comb(n, k)
        if a > X: break
        cnt[a] += 1 if 2*k == n else 2
        n += 1
    k += 1
N = {a: c + 2 for a, c in cnt.items()}
print('a <= 1e15 with N(a) >= 6:', sorted((a, c) for a, c in N.items() if c >= 6))
F = [0, 1]
while len(F) < 30: F.append(F[-1] + F[-2])
for i in range(1, 6):
    n = F[2*i+2]*F[2*i+3] - 1; k = F[2*i]*F[2*i+3] - 1
    print(i, n, k, comb(n+1, k+1) == comb(n, k+2), len(str(comb(n, k+2))), 'digits')

a <= 1e15 with N(a) >= 6: [(120, 6), (210, 6), (1540, 6), (3003, 8), (7140, 6), (11628, 6), (24310, 6)]
1 14 4 True 4 digits
2 103 38 True 29 digits
3 713 271 True 205 digits
4 4894 1868 True 1412 digits
5 33551 12814 True 9688 digits


The numbers with $N(a)=6$ below $10^{15}$ are a short list, $3003$ is the only one with $8$, and the family of Proposition 8.2 is there (its second member, with $29$ digits, is beyond our range). The others are sporadic coincidences, such as $120=\binom{16}2=\binom{10}3$ and $1540=\binom{56}2=\binom{22}3$; finding all solutions of $\binom nk=\binom m l$ is itself a research problem (de Weger, 1996; Blokhuis, Brouwer and de Weger, 2017).

**P11.2.** At $t=\frac m{v_i+v_j}$ runners $i$ and $j$ are at symmetric positions ($v_it+v_jt$ is an integer), which is the configuration used in the proof of Proposition 8.3; for the purpose of *verification*, any $t$ that works is enough, so a heuristic list of candidates is fine.

In [3]:
from itertools import combinations
def dist(x): x = x % 1; return min(x, 1 - x)
def best_time(v):
    best = (Fr(-1), None)
    for i in range(len(v)):
        for j in range(i, len(v)):
            s = v[i] + v[j]
            for m in range(1, s):
                t = Fr(m, s); val = min(dist(vi * t) for vi in v)
                if val > best[0]: best = (val, t)
    return best
for k in range(1, 5):
    worst, tight = Fr(1), []
    for v in combinations(range(1, 13), k):
        if math.gcd(*v) != 1: continue
        val, t = best_time(v)
        assert val >= Fr(1, k+1), v
        if val == Fr(1, k+1): tight.append(v)
        worst = min(worst, val)
    print(f'k={k}: all speed sets OK; worst {worst}; tight sets: {tight[:6]}{" ..." if len(tight) > 6 else ""} ({len(tight)})')

k=1: all speed sets OK; worst 1/2; tight sets: [(1,)] (1)
k=2: all speed sets OK; worst 1/3; tight sets: [(1, 2)] (1)


k=3: all speed sets OK; worst 1/4; tight sets: [(1, 2, 3)] (1)


k=4: all speed sets OK; worst 1/5; tight sets: [(1, 2, 3, 4), (1, 3, 4, 7)] (2)


Every speed set passes, and the tight sets include $\{1,2,\dots,k\}$ of Exercise 8.2; for $k=4$ there is a second one in this range, $\{1,3,4,7\}$. Tight speed sets are rare, and describing all of them for small $k$ is a nice project (with a proof that the candidate times suffice).

**P11.3.** Assign the differences $n,n-1,\dots,1$ in turn: difference $d$ must be realised by an edge with one endpoint already labelled (or by placing a new pair $\{x,x+d\}$). Labelling vertices one at a time, with the largest remaining difference forced first, prunes the search drastically.

In [4]:
def graceful(T):
    n = T.number_of_edges(); nodes = list(T.nodes)
    label, used = {}, set()
    diffs = set()
    def edges_ok(v):
        for u in T.neighbors(v):
            if u in label:
                d = abs(label[u] - label[v])
                if d == 0 or d in diffs: return None
        return True
    # order vertices by BFS from a vertex of maximal degree
    root = max(nodes, key=T.degree)
    order = list(nx.bfs_tree(T, root))
    def rec(idx):
        if idx == len(order): return True
        v = order[idx]
        for x in range(n + 1):
            if x in used: continue
            new = [abs(x - label[u]) for u in T.neighbors(v) if u in label]
            if len(set(new)) < len(new) or any(d in diffs or d == 0 for d in new): continue
            label[v] = x; used.add(x); diffs.update(new)
            if rec(idx + 1): return True
            del label[v]; used.discard(x); diffs.difference_update(new)
        return False
    return dict(label) if rec(0) else None
import time
for nv in range(2, 12):
    t0 = time.time(); trees = list(nx.nonisomorphic_trees(nv)); ok = sum(graceful(T) is not None for T in trees)
    print(f'{nv:2d} vertices: {len(trees):4d} trees, graceful: {ok:4d}   ({time.time()-t0:.1f}s)')
S = nx.Graph([('c','a1'),('a1','a2'),('c','b1'),('b1','b2'),('c','d1'),('d1','d2')])
print('spider:', graceful(S))

 2 vertices:    1 trees, graceful:    1   (0.0s)
 3 vertices:    1 trees, graceful:    1   (0.0s)
 4 vertices:    2 trees, graceful:    2   (0.0s)
 5 vertices:    3 trees, graceful:    3   (0.0s)
 6 vertices:    6 trees, graceful:    6   (0.0s)
 7 vertices:   11 trees, graceful:   11   (0.6s)


 8 vertices:   23 trees, graceful:   23   (0.1s)


 9 vertices:   47 trees, graceful:   47   (0.4s)


10 vertices:  106 trees, graceful:  106   (4.8s)


11 vertices:  235 trees, graceful:  235   (54.8s)
spider: {'c': 0, 'a1': 1, 'b1': 5, 'd1': 6, 'a2': 4, 'b2': 3, 'd2': 2}


All trees up to $11$ vertices are graceful, as expected (the known verification goes to $35$). Backtracking is exponential in the worst case, and the art of the large searches is in the order of the choices and the pruning.

**P11.4.**

In [5]:
from itertools import combinations as comb_
def count_sub(G, kind):
    if kind == 'edge': return G.number_of_edges()
    if kind == 'triangle': return sum(nx.triangles(G).values()) // 3
    if kind == 'P3': return sum(comb(d, 2) for _, d in G.degree())
sizes = {'edge': 2, 'triangle': 3, 'P3': 3}
for trial in range(5):
    n = 9; G = nx.gnp_random_graph(n, 0.45, seed=int(rng.integers(10**6)))
    deck = [G.subgraph([u for u in G if u != v]).copy() for v in G]
    row = []
    for kind, f in sizes.items():
        lhs = count_sub(G, kind); rhs = Fr(sum(count_sub(H, kind) for H in deck), n - f)
        row.append(f'{kind}: {lhs} = {rhs}')
    print(' | '.join(row))

edge: 18 = 18 | triangle: 12 = 12 | P3: 64 = 64
edge: 15 = 15 | triangle: 4 = 4 | P3: 39 = 39
edge: 15 = 15 | triangle: 3 = 3 | P3: 42 = 42
edge: 15 = 15 | triangle: 7 = 7 | P3: 48 = 48
edge: 16 = 16 | triangle: 6 = 6 | P3: 49 = 49


**P11.5.** A sieve for $\varphi$ up to $10^7$ with numpy, then a direct test.

In [6]:
M = 10**7
phi = np.arange(M + 1, dtype=np.int64)
for p in range(2, M + 1):
    if phi[p] == p:                       # p is prime
        phi[p::p] -= phi[p::p] // p
n = np.arange(M + 1, dtype=np.int64)
prime = phi == n - 1
comp = (n >= 4) & ~prime
lehmer = np.nonzero(comp & ((n - 1) % np.maximum(phi, 1) == 0))[0]
print('composite n <= 1e7 with phi(n) | n-1:', lehmer.tolist())
from sympy import factorint
# Korselt: squarefree, composite, p-1 | n-1; a Fermat test with base 2 filters first
cand = np.nonzero(comp & (n % 2 == 1))[0]
carm = [int(m) for m in cand if pow(2, int(m) - 1, int(m)) == 1 and all(e == 1 and (int(m) - 1) % (p - 1) == 0 for p, e in factorint(int(m)).items())]
print(len(carm), 'Carmichael numbers below 1e7; first ones:', carm[:8])
ratios = sorted(((Fr(c - 1, int(phi[c])), c) for c in carm), key=lambda r: abs(r[0] - round(r[0])))
print('closest (n-1)/phi(n) to an integer:', [(str(r), c) for r, c in ratios[:5]])

composite n <= 1e7 with phi(n) | n-1: []


105 Carmichael numbers below 1e7; first ones: [561, 1105, 1729, 2465, 2821, 6601, 8911, 10585]
closest (n-1)/phi(n) to an integer: [('2969/2916', 6733693), ('638/625', 3828001), ('46/45', 9439201), ('307/300', 6189121), ('2779/2704', 4335241)]


No Lehmer number, of course: one would need at least fourteen prime factors (Cohen and Hagis), hence be enormous. The Carmichael numbers satisfy every condition of Proposition 8.8 except the main one, and the ratios $(n-1)/\varphi(n)$ show how far they stay from being integers.

**P11.6.**

In [7]:
f, sols = 1, []
for m in range(1, 2001):
    f *= m
    r = isqrt(f + 1)
    if r * r == f + 1: sols.append((m, r))
print('n <= 2000 with n!+1 a square:', sols)
from sympy import primerange
ok = all((math.factorial(k) * math.factorial(p - 1 - k) - (-1)**(k+1)) % p == 0 for p in primerange(2, 200) for k in range(p))
print('Proposition wilsonflip holds for all p < 200:', ok)

n <= 2000 with n!+1 a square: [(4, 5), (5, 11), (7, 71)]
Proposition wilsonflip holds for all p < 200: True


**P11.7.**

In [8]:
def sendov_gap(z):
    c = np.poly(z); w = np.roots(np.polyder(c))
    return max(min(abs(a - w)) for a in z)
for n in (3, 5, 8, 12):
    disc = [sendov_gap(np.sqrt(rng.uniform(0, 1, n)) * np.exp(2j*np.pi*rng.uniform(0, 1, n))) for _ in range(2000)]
    circ = [sendov_gap(np.exp(2j*np.pi*rng.uniform(0, 1, n))) for _ in range(2000)]
    roots1 = np.exp(2j*np.pi*np.arange(n)/n)
    pert = [sendov_gap(roots1 * (1 - 0.01*rng.uniform(0, 1, n)) * np.exp(0.01j*rng.normal(size=n))) for _ in range(500)]
    print(f'n={n:2d}: uniform in disc max {max(disc):.3f} | on circle max {max(circ):.3f} | near z^n-1 max {max(pert):.4f} | bound 2n^(-1/(n-1)) = {2*n**(-1/(n-1)):.3f}')

n= 3: uniform in disc max 0.825 | on circle max 0.989 | near z^n-1 max 0.9971 | bound 2n^(-1/(n-1)) = 1.155


n= 5: uniform in disc max 0.542 | on circle max 0.697 | near z^n-1 max 0.9062 | bound 2n^(-1/(n-1)) = 1.337


n= 8: uniform in disc max 0.452 | on circle max 0.482 | near z^n-1 max 0.7202 | bound 2n^(-1/(n-1)) = 1.486


n=12: uniform in disc max 0.442 | on circle max 0.322 | near z^n-1 max 0.5598 | bound 2n^(-1/(n-1)) = 1.596


Random polynomials stay well inside Sendov's bound. The extreme case is $z^n-1$, where the gap is exactly $1$ (Exercise 8.6); small perturbations stay close to $1$ in low degree, but for larger $n$ they split the $(n-1)$-fold critical point $0$ and the gap drops quickly. The weak bound of Proposition 8.11 is above $1$ for $n\ge3$ and tends to $2$: the distance between the easy result and the conjecture is the whole problem.